In [ ]:
import os
import sys
from pathlib import Path

root_env = os.environ.get("CDR4176_ROOT")
if root_env:
    repo_root = Path(root_env).resolve()
else:
    repo_root = Path.cwd().resolve()
    while repo_root != repo_root.parent:
        if (repo_root / "project.yml").exists() or (repo_root / "CDR4176-main").is_dir():
            break
        repo_root = repo_root.parent
    if not ((repo_root / "project.yml").exists() or (repo_root / "CDR4176-main").is_dir()):
        raise RuntimeError("Could not locate repository root containing project.yml or CDR4176-main; set CDR4176_ROOT")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scripts.notebook_data import resolve_raw_path

NOTEBOOK_ID = "verification/scripts/results_simulations_3to7thermo_deco_tran.ipynb"

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def readRaw(rawfile: str, variables: list[str]) -> dict[str,list[float]]:
    """
    readRaw()
    =======
    Lee un archivo raw y devuelve las variables indicadas en un
    diccionario. Las llaves son el nombre de las variables en
    la simulacion, y los valores son listas que contienen el resultado
    de simulacion de su respectiva variable.

    @author: pdominguez. Contactense ante cualquier duda.

    Parameters
    ----------

    rawfile: str
             Archivo donde se almacenan las salidas de simulacion de ngspice. 
             El archivo debe estar configurado en filetype = ascii.
                    
    variables: list[str]
               Lista que contiene las llaves del diccionario de salida.
               Las llaves deben ser iguales a los nombres de las senales en la salida ascii.

    Returns
    -------

    dict[str,list[float]]
            Diccionario donde las llaves son strings iguales al nombre de la senal en simulacion, 
            y el valor es la lista de floats que contiene los valores simulados respectivos a esa 
            variable
    """
    out_dict = {}
    aux_dict = {}
    head = None

    raw_path = resolve_raw_path(rawfile, NOTEBOOK_ID)
    with open(raw_path, 'r') as f:
        for line in f:
            linea = line.strip()
            #Flag para identificar la cabecera con las variables
            if(linea == 'Variables:'):
                head = True
            elif(linea == 'Values:'):
                head = False
            
            #Procesamiento de la cabecera y los datos
            if(head == True and linea != 'Variables:'):
                lin_split = linea.split()
                if lin_split[1] in variables:
                    aux_dict[int(lin_split[0])] = lin_split[1]
                    out_dict[lin_split[1]] = []

            elif(head == False and linea != 'Values:'):
                if(len(linea.split()) > 1):  #Nuevo instante de simulacion
                    offset = 0
                if offset in aux_dict:
                    if (offset != 0):
                        label = aux_dict[offset]
                        out_dict[label].append(float(linea))
                    else:
                        label = aux_dict[offset]
                        out_dict[label].append(float(linea.split()[1]))
                offset = offset + 1
    
    return out_dict

In [ ]:
variables = ['time', 'v(vo_st7)','v(vo_st6)','v(vo_st5)', 'v(vo_st4)', 'v(vo_st3)', 'v(vo_st2)', 'v(vo_st1)']
resultados1 = readRaw('out1.raw',variables)
resultados2 = readRaw('out2.raw',variables)
resultados3 = readRaw('out3.raw',variables)
resultados4 = readRaw('out4.raw',variables)
resultados5 = readRaw('out5.raw',variables)
resultados6 = readRaw('out6.raw',variables)
resultados7 = readRaw('out7.raw',variables)
vector_tiempo = resultados1['time']

In [ ]:
import matplotlib.pyplot as plt

todos_los_resultados = [resultados1, resultados2, resultados3, resultados4, resultados5, resultados6, resultados7]
# Nombres base que buscas
señales_buscadas = ['v(vo_st1)', 'v(vo_st2)', 'v(vo_st3)', 'v(vo_st4)', 'v(vo_st5)', 'v(vo_st6)', 'v(vo_st7)']

fig, ax = plt.subplots(figsize=[12, 7])

for i, res in enumerate(todos_los_resultados):
    # Obtenemos todas las llaves reales que hay en este archivo (ej: 'V(VO_ST1)')
    llaves_reales = res.keys()
    
    for s in señales_buscadas:
        # Buscamos si nuestra señal existe en las llaves reales (sin importar mayúsculas)
        encontrada = False
        for k in llaves_reales:
            if s.lower() == k.lower():
                ax.plot(res['time'], res[k], linewidth=0.5)
                encontrada = True
                break

# Configuraciones
ax.set_title('Curvas tb_3to7_deco-tran', fontsize=16)
ax.set_xlim([2e-10, 4e-10]) 
ax.grid(True, linestyle=':', alpha=0.5)
plt.show()